# 헤드 비교 — NCM / logreg / Tip-Adapter (feature service 기반, torch-free)

feature 를 서버에서 **한 번** 뽑아, 여러 분류 헤드를 나란히 비교하고 Tip-Adapter β를 스윕한다.
→ torch 불필요 (numpy + sklearn + matplotlib).

> ⚠️ **feature_service 가 켜져 있어야 동작.** `EM_CKPT=... bash scripts/serve_features.sh`

## 0. 임포트 (torch 없음)

In [ ]:
import os, sys, json, urllib.request
from pathlib import Path
_HERE = Path.cwd(); _REPO = _HERE
while not (_REPO / 'inspection').exists() and _REPO != _REPO.parent:
    _REPO = _REPO.parent
if str(_REPO) not in sys.path:
    sys.path.insert(0, str(_REPO))
import numpy as np
import matplotlib.pyplot as plt
from inspection.service.client_example import get_features
from inspection.em_classifier import ClassifierHead, TipAdapterHead
print('torch-free OK, repo:', _REPO)

## 1. 설정

In [ ]:
SERVER    = 'http://localhost:8000'   # feature_service 주소
DATA_ROOT = '/path/to/class_folders'  # FILL_IN (classA/ classB/ ...)
VAL_FRAC  = 0.3
SEED      = 0
BETAS     = [1, 2, 3, 5, 8, 12, 20]   # Tip-Adapter β 스윕
KNN_K     = 5
IMG_EXTS  = {'.png', '.jpg', '.jpeg', '.tif', '.tiff', '.bmp'}

## 2. 서비스 상태

In [ ]:
with urllib.request.urlopen(SERVER.rstrip('/') + '/health', timeout=30) as r:
    info = json.loads(r.read().decode())
model_info = info.get('model', {})
print('status:', info.get('status'), '| model:', model_info)
assert info.get('status') == 'ok', 'serve_features.sh 로 서비스 먼저 기동'

## 3. feature 추출 (서버에서 한 번)
클래스별 폴더 이미지를 `/features` 로 보내 embedding 수집 → X, y.

In [ ]:
root = Path(DATA_ROOT)
class_dirs = sorted([d for d in root.iterdir() if d.is_dir()])
assert class_dirs, f'클래스 폴더 없음: {root}'
class_names = [d.name for d in class_dirs]
X_list, y_list = [], []
for i, d in enumerate(class_dirs):
    paths = [str(p) for p in sorted(d.rglob('*')) if p.suffix.lower() in IMG_EXTS]
    feats, _ = get_features(SERVER, paths, feature_kind=model_info.get('feature_kind'))
    X_list.append(feats); y_list.append(np.full(len(feats), i))
    print(f'  {d.name}: {len(paths)}장')
X = np.concatenate(X_list); y = np.concatenate(y_list)
print('X', X.shape, '| classes', class_names)

## 4. t-SNE — 클래스가 여러 군집으로 갈리나? (multi-modal 진단)
한 클래스가 여러 덩어리로 흩어지면 NCM(평균)이 불리, tip/kNN 이 유리하다는 신호.

In [ ]:
try:
    from sklearn.manifold import TSNE
    emb = TSNE(n_components=2, init='pca',
               perplexity=min(30, max(5, len(X)//4))).fit_transform(X)
    plt.figure(figsize=(7, 6))
    for i, name in enumerate(class_names):
        m = y == i
        plt.scatter(emb[m, 0], emb[m, 1], s=12, alpha=0.7, label=name)
    plt.legend(markerscale=2, fontsize=8); plt.title('feature t-SNE (클래스별)')
    plt.tight_layout(); plt.show()
except Exception as e:
    print('t-SNE 스킵:', e)

## 5. 헤드 비교 (train/val) + Tip-Adapter β 스윕
같은 feature·같은 split 에서 NCM / kNN / logreg / tip 정확도 비교.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
tr, va = train_test_split(np.arange(len(y)), test_size=VAL_FRAC, stratify=y, random_state=SEED)
Xtr, ytr, Xva, yva = X[tr], y[tr], X[va], y[va]

def acc(head):
    return float((head.predict(Xva) == yva).mean())

results = {}
results['ncm']    = acc(ClassifierHead.fit(Xtr, ytr, class_names, kind='ncm'))
results['logreg'] = acc(ClassifierHead.fit(Xtr, ytr, class_names, kind='logreg'))
knn = KNeighborsClassifier(n_neighbors=min(KNN_K, len(Xtr)), metric='cosine').fit(Xtr, ytr)
results['knn']    = float((knn.predict(Xva) == yva).mean())

tip_sweep = {b: acc(TipAdapterHead.fit(Xtr, ytr, class_names, beta=b)) for b in BETAS}
best_b = max(tip_sweep, key=tip_sweep.get)
results[f'tip(β={best_b})'] = tip_sweep[best_b]

print('=== val accuracy ===')
for k, v in results.items():
    print(f'  {k:14s}: {v*100:.1f}%')
print('tip β 스윕:', {b: round(a*100, 1) for b, a in tip_sweep.items()})

In [ ]:
# 시각화: β 스윕 곡선 + 헤드 막대
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(list(tip_sweep), [a*100 for a in tip_sweep.values()], 'o-')
ax[0].axhline(results['ncm']*100, ls='--', c='gray', label='ncm')
ax[0].axhline(results['logreg']*100, ls='--', c='green', label='logreg')
ax[0].set_xlabel('Tip-Adapter β'); ax[0].set_ylabel('val acc (%)')
ax[0].set_title('tip β 스윕'); ax[0].legend()
names_ = list(results); vals_ = [results[k]*100 for k in names_]
ax[1].bar(names_, vals_); ax[1].set_ylabel('val acc (%)'); ax[1].set_title('헤드 비교')
for i, v in enumerate(vals_): ax[1].text(i, v, f'{v:.1f}', ha='center', va='bottom', fontsize=9)
plt.setp(ax[1].get_xticklabels(), rotation=20, ha='right')
plt.tight_layout(); plt.show()

## 6. 최고 헤드 혼동행렬

In [ ]:
best_name = max(results, key=results.get)
if best_name.startswith('tip'):
    best = TipAdapterHead.fit(Xtr, ytr, class_names, beta=best_b)
elif best_name == 'knn':
    best = None  # knn 은 sklearn 객체
else:
    best = ClassifierHead.fit(Xtr, ytr, class_names, kind=best_name)
pred = knn.predict(Xva) if best is None else best.predict(Xva)
C = len(class_names); cm = np.zeros((C, C), int)
for p, t in zip(pred, yva): cm[t, p] += 1
fig, axc = plt.subplots(figsize=(1.2*C+2, 1.2*C+1))
axc.imshow(cm, cmap='Blues')
axc.set_xticks(range(C)); axc.set_xticklabels(class_names, rotation=45, ha='right')
axc.set_yticks(range(C)); axc.set_yticklabels(class_names)
axc.set_xlabel('pred'); axc.set_ylabel('true'); axc.set_title(f'best={best_name} ({results[best_name]*100:.1f}%)')
for i in range(C):
    for j in range(C): axc.text(j, i, cm[i, j], ha='center', va='center')
plt.tight_layout(); plt.show()

## 7. 선택한 헤드 저장 (전체 데이터로 재학습)
배포용은 val 로 고른 방식으로 **전체 데이터** 재학습해 저장.

In [ ]:
OUT = './out/best_head.npz'
os.makedirs(Path(OUT).parent, exist_ok=True)
meta = {'server': SERVER, **model_info}
if best_name.startswith('tip'):
    head = TipAdapterHead.fit(X, y, class_names, beta=best_b, meta=meta)
elif best_name == 'knn':
    head = TipAdapterHead.fit(X, y, class_names, beta=best_b, meta=meta)  # knn≈tip(고β), tip 로 저장
else:
    head = ClassifierHead.fit(X, y, class_names, kind=best_name, meta=meta)
head.save(OUT)
print('saved:', OUT, '| kind:', best_name)
# 추론: client_example.py classify --head out/best_head.npz  (또는 ClassifierHead/TipAdapterHead.load)

## 참고 — 해석 가이드
- **NCM 낮은데 tip/kNN 높음** → 클래스가 여러 하위 군집(multi-modal). §4 t-SNE 에서 확인.
- **logreg 가 최고** → 선형 분리로 충분(단봉 클래스).
- **β**: 크면 하드 최근접(NN)에 근접, 작으면 평탄(평균에 근접). 스윕에서 꼭대기 근처 선택.